In [ ]:
import pandas as pd
import re
from bert_score import score

gold = pd.read_csv("correct_answers.csv")
model1 = pd.read_csv("model1_results.csv")
model2 = pd.read_csv("model2_results.csv")

# Clean column names
gold.columns = gold.columns.str.strip()
model1.columns = model1.columns.str.strip()
model2.columns = model2.columns.str.strip()

# Standardize columns
gold = gold.rename(columns={"correct_answers": "gold", "correct_answer": "gold"})
model1 = model1.rename(columns={"answer": "model1"})
model2 = model2.rename(columns={"answer": "model2"})

# Merge all
df = gold.merge(model1, on="id").merge(model2, on="id")

def extract_laws(text):
    return set(re.findall(r"§\s*\d+\s*[A-Za-z]*", str(text)))

def jaccard(a, b):
    if len(a) == 0 and len(b) == 0:
        return 1.0
    return len(a & b) / len(a | b)

law_score_m1 = []
law_score_m2 = []

for g, m1, m2 in zip(df["gold"], df["model1"], df["model2"]):
    g_laws = extract_laws(g)

    law_score_m1.append(jaccard(g_laws, extract_laws(m1)))
    law_score_m2.append(jaccard(g_laws, extract_laws(m2)))

df["law_score_m1"] = law_score_m1
df["law_score_m2"] = law_score_m2

_, _, f1_m1 = score(df["model1"].tolist(), df["gold"].tolist(), lang="de")
_, _, f1_m2 = score(df["model2"].tolist(), df["gold"].tolist(), lang="de")

df["bert_m1"] = f1_m1.tolist()
df["bert_m2"] = f1_m2.tolist()

df["final_m1"] = 0.3 * df["law_score_m1"] + 0.7 * df["bert_m1"]
df["final_m2"] = 0.3* df["law_score_m2"] + 0.7 * df["bert_m2"]

results = pd.DataFrame({
    "Model": ["Model 1", "Model 2"],
    "Avg Law Score": [df["law_score_m1"].mean(), df["law_score_m2"].mean()],
    "Avg BERTScore": [df["bert_m1"].mean(), df["bert_m2"].mean()],
    "Final Score": [df["final_m1"].mean(), df["final_m2"].mean()],
})

print(results)

results.to_csv("model_comparison_results2.csv", index=False)